<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/txt_to_parquet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Txt a Parquet - Tesis 2026

In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.colab import drive
import os

# Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Path
file_path = '/content/drive/MyDrive/Tesis2026/deudores_febrero.txt'

# Configuración basada en el PDF (Anchos y Nombres)
anchos = [5, 6, 2, 11, 3, 2, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1, 1, 1, 1, 1, 1, 4]
nombres = [
    "cod_entidad", "fecha", "tipo_id", "nro_id", "actividad", "situacion",
    "prestamos", "sin_uso", "garantias_otorg", "otros_conceptos",
    "garantia_pref_A", "garantia_pref_B", "sin_garantia_pref",
    "contragar_pref_A", "contragar_pref_B", "sin_contragar_pref",
    "previsiones", "deuda_cubierta", "proceso_judicial",
    "refinanciaciones", "recat_obligatoria", "sit_juridica", "irrecuperable", "dias_atraso"
]

col_string  = ["cod_entidad", "fecha", "tipo_id", "nro_id", "situacion", "deuda_cubierta",
               "proceso_judicial", "refinanciaciones", "recat_obligatoria", "sit_juridica", "irrecuperable"]
col_numeric = ["actividad", "prestamos", "sin_uso", "garantias_otorg", "otros_conceptos",
               "garantia_pref_A", "garantia_pref_B", "sin_garantia_pref",
               "contragar_pref_A", "contragar_pref_B", "sin_contragar_pref",
               "previsiones", "dias_atraso"]

try:
    reader = pd.read_fwf(file_path, widths=anchos, names=nombres, chunksize=100000, engine='python', dtype=str)
    writer = None
    total_rows = 0

    print("Procesando según especificaciones del PDF...")
    for i, chunk in enumerate(reader):

        # Columnas numéricas: strip + reemplazar coma decimal + convertir
        for col in col_numeric:
            chunk[col] = pd.to_numeric(
                chunk[col].str.strip().str.replace(',', '.', regex=False),
                errors='coerce'
            ).fillna(0).astype('float64')

        # Columnas string: strip + rellenar nulos
        for col in col_string:
            chunk[col] = chunk[col].str.strip().fillna('').astype(str)

        table = pa.Table.from_pandas(chunk)

        if writer is None:
            writer = pq.ParquetWriter(
                '/content/drive/MyDrive/Tesis2026/deudores_final_febrero.parquet',
                table.schema
            )

        writer.write_table(table)
        total_rows += len(chunk)
        if (i + 1) % 10 == 0:
            print(f"Bloque {i+1} OK. Filas: {total_rows:,}")

except Exception as e:
    print(f"\nError: {e}")
finally:
    if writer is not None:
        writer.close()
        print("\nArchivo Parquet finalizado.")

print("Fin del proceso.")

Procesando según especificaciones del PDF...
Bloque 10 OK. Filas: 1,000,000
Bloque 20 OK. Filas: 2,000,000
Bloque 30 OK. Filas: 3,000,000
Bloque 40 OK. Filas: 4,000,000
Bloque 50 OK. Filas: 5,000,000
Bloque 60 OK. Filas: 6,000,000
Bloque 70 OK. Filas: 7,000,000
Bloque 80 OK. Filas: 8,000,000
Bloque 90 OK. Filas: 9,000,000
Bloque 100 OK. Filas: 10,000,000
Bloque 110 OK. Filas: 11,000,000
Bloque 120 OK. Filas: 12,000,000
Bloque 130 OK. Filas: 13,000,000
Bloque 140 OK. Filas: 14,000,000
Bloque 150 OK. Filas: 15,000,000
Bloque 160 OK. Filas: 16,000,000
Bloque 170 OK. Filas: 17,000,000
Bloque 180 OK. Filas: 18,000,000
Bloque 190 OK. Filas: 19,000,000
Bloque 200 OK. Filas: 20,000,000
Bloque 210 OK. Filas: 21,000,000
Bloque 220 OK. Filas: 22,000,000
Bloque 230 OK. Filas: 23,000,000
Bloque 240 OK. Filas: 24,000,000
Bloque 250 OK. Filas: 25,000,000
Bloque 260 OK. Filas: 26,000,000
Bloque 270 OK. Filas: 27,000,000
Bloque 280 OK. Filas: 28,000,000
Bloque 290 OK. Filas: 29,000,000
Bloque 300 OK. F